# Word Embeddings and Language Models
### Interactive Notebook for AI/ML Interview Preparation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from collections import Counter, defaultdict
np.random.seed(42)
print('Libraries loaded!')

---
## 1. Co-occurrence Matrix & SVD Embeddings

In [ ]:
# Build embeddings from co-occurrence matrix
corpus = [
    'king queen royal throne palace',
    'man woman child family home',
    'king man prince royal heir',
    'queen woman princess royal crown',
    'dog cat pet animal house',
    'cat kitten pet small cute',
    'dog puppy pet loyal friend',
]
docs = [s.split() for s in corpus]
vocab = sorted(set(w for d in docs for w in d))
w2i = {w:i for i,w in enumerate(vocab)}

# Co-occurrence within window=2
cooccur = np.zeros((len(vocab), len(vocab)))
for doc in docs:
    for i, w in enumerate(doc):
        for j in range(max(0,i-2), min(len(doc),i+3)):
            if i != j: cooccur[w2i[w], w2i[doc[j]]] += 1

# SVD to get embeddings
U, s, Vt = np.linalg.svd(cooccur)
embeddings = U[:, :3] * s[:3]  # 3-dim embeddings
print(f'Vocabulary: {len(vocab)} words → {embeddings.shape[1]}-dim embeddings')

In [ ]:
# Visualize in 2D with PCA
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings)

plt.figure(figsize=(8, 6))
for i, word in enumerate(vocab):
    plt.scatter(emb_2d[i, 0], emb_2d[i, 1], s=50)
    plt.annotate(word, (emb_2d[i, 0]+0.05, emb_2d[i, 1]+0.05), fontsize=9)
plt.title('Word Embeddings (SVD on Co-occurrence)')
plt.tight_layout(); plt.show()
print('Similar words cluster together!')

---
## 2. N-gram Language Model

In [ ]:
# Bigram language model
text = 'the cat sat on the mat the cat ate the rat the dog chased the cat'
words = text.split()
bigrams = defaultdict(Counter)
for i in range(len(words)-1):
    bigrams[words[i]][words[i+1]] += 1

def predict_next(word, n=3):
    if word in bigrams:
        total = sum(bigrams[word].values())
        probs = {w: c/total for w, c in bigrams[word].most_common(n)}
        return probs
    return {}

print('Bigram predictions:')
for w in ['the', 'cat', 'dog']:
    print(f'  After "{w}": {predict_next(w)}')

# Generate text
generated = ['the']
for _ in range(8):
    preds = predict_next(generated[-1])
    if preds:
        words_list = list(preds.keys())
        probs_list = list(preds.values())
        next_word = np.random.choice(words_list, p=probs_list)
        generated.append(next_word)
    else: break
print(f'\nGenerated: {" ".join(generated)}')

---
## 3. BPE Tokenization from Scratch

In [ ]:
def bpe_tokenize(text, n_merges=10):
    # Initialize: split each word into characters
    words = text.lower().split()
    tokens = [list(w) + ['</w>'] for w in words]
    
    for merge_step in range(n_merges):
        # Count pairs
        pairs = Counter()
        for token_list in tokens:
            for i in range(len(token_list)-1):
                pairs[(token_list[i], token_list[i+1])] += 1
        if not pairs: break
        best = pairs.most_common(1)[0][0]
        
        # Merge best pair
        new_tokens = []
        for token_list in tokens:
            merged = []
            i = 0
            while i < len(token_list):
                if i < len(token_list)-1 and (token_list[i], token_list[i+1]) == best:
                    merged.append(token_list[i] + token_list[i+1])
                    i += 2
                else:
                    merged.append(token_list[i])
                    i += 1
            new_tokens.append(merged)
        tokens = new_tokens
        if merge_step < 5:
            print(f'Merge {merge_step+1}: {best[0]} + {best[1]} → {best[0]+best[1]}')
    
    return tokens

result = bpe_tokenize('the cat sat on the mat the cat ate')
print(f'\nFinal tokens: {result[:4]}...')

---
## Key Interview Takeaways

1. **Word2Vec** — Skip-gram predicts context from word; CBOW predicts word from context
2. **Co-occurrence + SVD** — simple but effective embeddings (GloVe-like)
3. **N-gram models** — count-based; foundation for understanding neural LMs
4. **BPE** — subword tokenization; handles unknown words and morphology
5. **Perplexity** — lower is better; measures how well model predicts text

---

<small><em>© 2026 AI Nirvana · Disclaimer: Provided as is. No liability assumed.</em></small>